# Breast cancer classification with Spark ML and OMLE

Use Spark DataFrames throughout loading, type casting, splitting, training, evaluation, and parity checks. Score the exported pipeline through `omle-spark` on Spark executors; no pandas, NumPy, or scikit-learn code is used in this notebook.

## Setup and source

Install `requirements-spark.txt`, use Java 17 with `JAVA_HOME` configured, and start from a fresh notebook kernel. PySpark 4.3 requires Python 3.11+. The bundled OMLE JARs must be configured before Spark starts. Stop any existing Spark session and restart the kernel if it was created without these JARs.

[UCI Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic), Wolberg, Mangasarian, Street, and Street (1993), CC BY 4.0. Label 0 = malignant; label 1 = benign. This is an educational example, not a clinically validated model.

Download the original file into the ignored `.cache/spark/` directory, validate its SHA-256, and read it with Spark's CSV parser using an explicit source schema. The UCI source is a headerless CSV with an ID, diagnosis, and 30 numeric features.

This local example uses a local model path. For a cluster, place datasets and model artifacts on storage accessible to every executor. The final cell stops Spark.

In [ ]:
import gzip
import hashlib
import json
import os
import sys
from importlib.metadata import version
from pathlib import Path
from urllib.request import urlopen

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, SQLTransformer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator, RegressionEvaluator
from pyspark.ml.functions import vector_to_array

import omle
from omle_convert import export_omle
import omle_spark
from omle_spark import OMLEModel

SEED = 42
ATOL = 1e-4
RTOL = 1e-4
integer_columns = []
float_columns = ['mean_radius', 'mean_texture', 'mean_perimeter', 'mean_area', 'mean_smoothness', 'mean_compactness', 'mean_concavity', 'mean_concave_points', 'mean_symmetry', 'mean_fractal_dimension', 'se_radius', 'se_texture', 'se_perimeter', 'se_area', 'se_smoothness', 'se_compactness', 'se_concavity', 'se_concave_points', 'se_symmetry', 'se_fractal_dimension', 'worst_radius', 'worst_texture', 'worst_perimeter', 'worst_area', 'worst_smoothness', 'worst_compactness', 'worst_concavity', 'worst_concave_points', 'worst_symmetry', 'worst_fractal_dimension']
categorical_columns = []
feature_columns = integer_columns + float_columns + categorical_columns
source_columns = ['source_id', 'diagnosis', 'mean_radius', 'mean_texture', 'mean_perimeter', 'mean_area', 'mean_smoothness', 'mean_compactness', 'mean_concavity', 'mean_concave_points', 'mean_symmetry', 'mean_fractal_dimension', 'se_radius', 'se_texture', 'se_perimeter', 'se_area', 'se_smoothness', 'se_compactness', 'se_concavity', 'se_concave_points', 'se_symmetry', 'se_fractal_dimension', 'worst_radius', 'worst_texture', 'worst_perimeter', 'worst_area', 'worst_smoothness', 'worst_compactness', 'worst_concavity', 'worst_concave_points', 'worst_symmetry', 'worst_fractal_dimension']
SOURCE_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data'
SOURCE_SHA256 = 'd606af411f3e5be8a317a5a8b652b425aaf0ff38ca683d5327ffff94c3695f4a'
packages = ["pyspark", "omle", "omle-convert", "omle-spark", "omle-viewer"]
environment = {"python": sys.version, "packages": {p: version(p) for p in packages}}
print(json.dumps(environment, indent=2))
cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents]
                  if (p / "requirements.txt").is_file() and (p / "notebooks").is_dir()), None)
if repo_root is None:
    raise RuntimeError("Start Jupyter from omle-models or a notebook subdirectory.")
os.environ["PYSPARK_PYTHON"] = sys.executable
spark = (SparkSession.builder.master("local[2]")
         .appName("omle-models-breast-cancer")
         .config("spark.jars", omle_spark.jars_classpath())
         .config("spark.ui.enabled", "false")
         .config("spark.sql.shuffle.partitions", "2")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

## Download and load with Spark

Keep file downloading and checksum verification in Python’s standard library; all parsing and data preparation use Spark. The checksum pins the exact source bytes. Cached data is reused only if its checksum matches.

In [ ]:
cache_dir = repo_root / ".cache" / "spark"
cache_dir.mkdir(parents=True, exist_ok=True)
source_path = cache_dir / 'wdbc.data'
if not source_path.exists():
    with urlopen(SOURCE_URL, timeout=120) as response:
        content = response.read()
    if content.startswith(b"\x1f\x8b"):
        content = gzip.decompress(content)
    if hashlib.sha256(content).hexdigest() != SOURCE_SHA256:
        raise ValueError("Source checksum changed; review the dataset before using it.")
    source_path.write_bytes(content)
if hashlib.sha256(source_path.read_bytes()).hexdigest() != SOURCE_SHA256:
    raise ValueError("Cached source checksum does not match the pinned dataset.")
source_schema = StructType([StructField(name, StringType(), True) for name in source_columns])
raw = spark.read.schema(source_schema).option("mode", "FAILFAST").csv(str(source_path))
assert raw.count() == 569, "Unexpected source row count"
print("Source rows:", raw.count())

## Cast and inspect inputs

Cast numeric and categorical columns explicitly in Spark. Treat `?` and blank categorical values as nulls. Preserve integer counts and floating-point measurements. Inspect the raw input contract and missing-value counts before fitting.

In [ ]:
typed = raw.select(
    *[F.col(name).cast("long").alias(name) for name in integer_columns],
    *[F.col(name).cast("double").alias(name) for name in float_columns],
    *[F.when(F.trim(F.col(name)).isin("?", ""), F.lit(None).cast("string"))
       .otherwise(F.trim(F.col(name))).alias(name) for name in categorical_columns],
    F.when(F.col("diagnosis") == "M", F.lit(0.0)).when(F.col("diagnosis") == "B", F.lit(1.0)).alias("label"),
)
assert typed.where(F.col("label").isNull()).count() == 0
if integer_columns:
    assert typed.where(" OR ".join(f"`{name}` IS NULL" for name in integer_columns)).count() == 0
# zipWithIndex retains duplicate records while assigning a unique stable ID
# for this checksum-pinned, locally read source and its partition order.
data = spark.createDataFrame(
    typed.rdd.zipWithIndex().map(lambda pair: tuple(pair[0]) + (int(pair[1]),)),
    schema=StructType(typed.schema.fields + [StructField("row_id", LongType(), False)]),
).cache()
data.printSchema()
data.show(5, truncate=False)
data.select(*[F.sum(F.col(name).isNull().cast("long")).alias(name) for name in feature_columns]).show()
assert data.select("row_id").distinct().count() == data.count()

## Split in Spark

Assign approximately 20% of rows to testing using a deterministic hash of the source row ID and seed. This Spark split is not stratified and does not reproduce the scikit-learn split. Each exported model is compared with its own native fitted pipeline. Cache the split once before fitting.

In [ ]:
partitioned = data.withColumn("test_bucket", F.pmod(F.xxhash64("row_id", F.lit(SEED)), F.lit(5)))
train_df = partitioned.where(F.col("test_bucket") != 0).drop("test_bucket").cache()
test_df = partitioned.where(F.col("test_bucket") == 0).drop("test_bucket").cache()
train_rows, test_rows = train_df.count(), test_df.count()
assert train_rows > 0 and test_rows > 0
assert train_rows + test_rows == data.count()
print(f"Training rows: {train_rows}; held-out rows: {test_rows}")
train_df.groupBy("label").count().orderBy("label").show()
test_df.groupBy("label").count().orderBy("label").show()
assert train_df.select("label").distinct().count() == 2
assert test_df.select("label").distinct().count() == 2

## Fit the Spark pipeline

Assemble the raw numeric columns, fit scaling on training rows only, and train logistic regression. All fitted preprocessing is included in the exported graph.

In [ ]:
stages = [VectorAssembler(inputCols=feature_columns, outputCol="assembled_features")]
stages += [StandardScaler(inputCol="assembled_features", outputCol="scaled_features", withMean=True, withStd=True)]
stages += [LogisticRegression(featuresCol="scaled_features", labelCol="label", maxIter=100, regParam=0.01)]
pipeline = Pipeline(stages=stages).fit(train_df)

## Evaluate with Spark ML

Evaluate held-out quality using Spark evaluators. Classification probability columns are ordered `[0, 1]`; AUC treats label 1 as positive. These quality metrics are independent from conversion fidelity.

In [ ]:
native_test = pipeline.transform(test_df).cache()
metrics = {
    name: float(MulticlassClassificationEvaluator(labelCol="label", metricName=name).evaluate(native_test))
    for name in ["accuracy", "f1", "logLoss"]
}
metrics["areaUnderROC_class_1"] = float(BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="probability", metricName="areaUnderROC"
).evaluate(native_test))
native_test.groupBy("label", "prediction").count().orderBy("label", "prediction").show()
print(json.dumps(metrics, indent=2))

## Export the fitted pipeline

Convert the entire fitted Spark pipeline and include training examples for embedded verification. All held-out and probe checks below score raw Spark columns with `omle-spark`. Rerunning overwrites only this recipe’s generated artifacts.

In [ ]:
output_dir = repo_root / "models" / "classification" / "breast-cancer"
output_dir.mkdir(parents=True, exist_ok=True)
model_path = output_dir / "spark-logistic-regression.omle"
export_omle(pipeline, str(model_path), dataset=train_df, feature_names=feature_columns,
            model_name="breast-cancer-spark-logistic-regression", model_version="0.2.0", random_state=SEED,
            n_verify=10, n_sample=3, verify_atol=ATOL, verify_rtol=RTOL)
print(f"Exported {model_path.relative_to(repo_root)} ({model_path.stat().st_size:,} bytes)")

## Inspect inputs, outputs, and operators

Inspect metadata from the saved artifact. Display the declarations in Spark tables; `-1` denotes variable batch size.

In [ ]:
exported_model = omle.load(str(model_path))
print("Inputs")
spark.createDataFrame([
    (s.name, s.type.dtype.name if s.type else "unspecified", json.dumps(s.type.shape) if s.type else "unspecified")
    for s in exported_model.inputs
], "name string, dtype string, shape string").show(truncate=False)
print("Outputs")
spark.createDataFrame([
    (s.name, s.type.dtype.name if s.type else "unspecified", json.dumps(s.type.shape) if s.type else "unspecified", s.role.name)
    for s in exported_model.outputs
], "name string, dtype string, shape string, role string").show(truncate=False)
operator_versions = {item.namespace: item.version for item in exported_model.operator_imports}
def operator_rows(nodes, parent=""):
    for node in nodes:
        path = f"{parent}/{node.name}" if parent else node.name
        yield (path, node.domain, node.op, str(operator_versions.get(node.domain, "")))
        if node.composite is not None:
            yield from operator_rows(node.composite.nodes, path)
print("Operators")
spark.createDataFrame(list(operator_rows(exported_model.nodes)),
                      "node string, namespace string, operator string, version string").show(100, truncate=False)

## Validate the model

Stop on structural validation errors. Warnings remain visible. Numerical parity follows below.

In [ ]:
validation = omle.validate(exported_model)
print(validation)
if not validation.is_valid:
    raise ValueError(f"Invalid exported OMLE model:\n{validation}")

## View the interactive graph

Run this cell in a notebook editor to inspect the model. GitHub’s static preview does not execute the viewer. The model stays local.

In [ ]:
from omle_viewer import show

viewer = show(exported_model, height=650)

## Verify predictions in Spark

Use the native executor-side OMLE transformer and join predictions by the unique row ID. Compare all test rows with Spark expressions, including output shapes, finiteness, and numeric tolerances. Only the small aggregate report is returned to Python. No test rows are collected into pandas or NumPy.

Before OMLE scoring, numeric nulls are represented as IEEE NaN because the published Spark bridge reads numeric nulls as zero. This preserves missingness; imputation remains in the exported graph.

In [ ]:
omle_model = OMLEModel(modelPath=str(model_path), predictionCol="omle_prediction", probabilityCol="omle_probability")

def verify_predictions(frame):
    expected_rows = frame.count()
    assert expected_rows > 0
    assert frame.select("row_id").distinct().count() == expected_rows
    native = pipeline.transform(frame).select("row_id", F.col("prediction").alias("native_prediction"), vector_to_array("probability").alias("native_probability"))
    # Preserve missing numeric values at the JVM bridge: the published
    # omle-spark bridge reads null numbers as zero, but accepts IEEE NaN.
    # The exported Imputer still performs the learned replacement.
    omle_input = frame.na.fill(float("nan"), subset=float_columns)
    actual = omle_model.transform(omle_input).select("row_id", "omle_prediction", vector_to_array("omle_probability").alias("omle_probability"))
    compared = native.join(actual, "row_id", "inner").cache()
    try:
        assert compared.count() == expected_rows
        invalid_prediction = (F.col("omle_prediction").isNull() | F.isnan("omle_prediction")
                              | (F.abs(F.col("omle_prediction")) == float("inf")))
        assert compared.where(invalid_prediction).count() == 0
        mismatches = compared.where(F.col("native_prediction") != F.col("omle_prediction")).count()
        assert mismatches == 0, f"{mismatches} classification labels differ"
        pairs = F.arrays_zip("native_probability", "omle_probability")
        invalid_probability = F.exists(F.col("omle_probability"), lambda v:
            v.isNull() | F.isnan(v) | (v < 0) | (v > 1))
        outside_tolerance = F.exists(pairs, lambda p:
            F.abs(p["omle_probability"] - p["native_probability"])
            > F.lit(ATOL) + F.lit(RTOL) * F.abs(p["native_probability"]))
        probability_sum = F.aggregate(F.col("omle_probability"), F.lit(0.0), lambda a, v: a + v)
        assert compared.where((F.size("omle_probability") != 2) | (F.size("native_probability") != 2)
                              | invalid_probability | outside_tolerance
                              | (F.abs(probability_sum - 1.0) > ATOL)).count() == 0
        max_error = compared.select(F.array_max(F.transform(pairs, lambda p:
            F.abs(p["omle_probability"] - p["native_probability"]))).alias("error")).agg(F.max("error")).first()[0]
        return {"rows": expected_rows, "label_agreement": 1.0, "max_absolute_probability_error": float(max_error)}
    finally:
        compared.unpersist()

parity = verify_predictions(test_df)
parity.update({"atol": ATOL, "rtol": RTOL})
print(json.dumps(parity, indent=2))
print("PASS: native Spark and OMLE predictions match.")

## Save the run record

Record source identity and checksum, Spark schema, deterministic split rule, training configuration, quality, and conversion fidelity. Generated artifacts stay ignored by Git. Seeds do not guarantee identical fitted models across Spark versions.

In [ ]:
report = {
    "environment": environment,
    "source_url": SOURCE_URL, "source_sha256": SOURCE_SHA256,
    "seed": SEED, "split_rule": "pmod(xxhash64(row_id, seed), 5) == 0 for test",
    "row_id_rule": "zipWithIndex on parsed checksum-pinned source rows",
    "train_rows": train_rows, "test_rows": test_rows,
    "feature_columns": feature_columns, "input_schema": data.schema.jsonValue(),
    "model_parameters": {f"{i}:{stage.__class__.__name__}.{param.name}": str(value)
                         for i, stage in enumerate(pipeline.stages)
                         for param, value in stage.extractParamMap().items()},
    "metrics": metrics, "parity": parity,
}
report_path = output_dir / "spark-logistic-regression-environment.json"
report_path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(f"Saved {report_path.relative_to(repo_root)}")

## Stop Spark

Release cached data and the local JVM. If an earlier cell failed, run this cell manually before restarting the kernel.

In [ ]:
spark.catalog.clearCache()
spark.stop()